In [1]:
import sqlite3
import pandas as pd
from pathlib import Path

2 — Database path

In [2]:
DB_PATH = Path("../database/fraud_detection.db")

print("Database path:", DB_PATH)
print("Database exists:", DB_PATH.exists())

Database path: ..\database\fraud_detection.db
Database exists: True


3 — Connect database

In [3]:
conn = sqlite3.connect(DB_PATH)

print("SQLite database connected successfully.")

SQLite database connected successfully.


4 — Create fraud_alerts table

In [4]:
create_alert_table_query = """
CREATE TABLE IF NOT EXISTS fraud_alerts (
    alert_id INTEGER PRIMARY KEY AUTOINCREMENT,
    created_at TEXT NOT NULL,
    transaction_amount REAL NOT NULL,
    merchant_category TEXT NOT NULL,
    payment_method TEXT NOT NULL,
    device_type TEXT NOT NULL,
    location TEXT NOT NULL,
    is_international INTEGER NOT NULL,
    suspicious_keyword TEXT NOT NULL,
    fraud_probability REAL NOT NULL,
    decision_threshold REAL NOT NULL,
    prediction INTEGER NOT NULL,
    status TEXT NOT NULL DEFAULT 'Open'
)
"""

conn.execute(create_alert_table_query)
conn.commit()

print("fraud_alerts table created successfully.")

fraud_alerts table created successfully.


5 — Verify table

In [5]:
tables = pd.read_sql_query(
    """
    SELECT name
    FROM sqlite_master
    WHERE type = 'table'
    ORDER BY name
    """,
    conn
)

tables

,name
0,fraud_alerts
1,sqlite_sequence
2,transactions


6 — Check table structure

In [6]:
alert_schema = pd.read_sql_query(
    "PRAGMA table_info(fraud_alerts)",
    conn
)

alert_schema

,cid,name,type,notnull,dflt_value,pk
0,0,alert_id,INTEGER,0,NaN,1
1,1,created_at,TEXT,1,NaN,0
2,2,transaction_amount,REAL,1,NaN,0
3,3,merchant_category,TEXT,1,NaN,0
4,4,payment_method,TEXT,1,NaN,0
5,5,device_type,TEXT,1,NaN,0
6,6,location,TEXT,1,NaN,0
7,7,is_international,INTEGER,1,NaN,0
8,8,suspicious_keyword,TEXT,1,NaN,0
9,9,fraud_probability,REAL,1,NaN,0


7 — Check initially empty table

In [7]:
alert_count = pd.read_sql_query(
    "SELECT COUNT(*) AS total_alerts FROM fraud_alerts",
    conn
)

alert_count

,total_alerts
0,0


8 — Close connection

In [8]:
conn.close()

print("Database connection closed.")

Database connection closed.


9 — Reconnect database

In [9]:
conn = sqlite3.connect(DB_PATH)

print("Database reconnected successfully.")

Database reconnected successfully.


10 — Create Alert Function

In [16]:
def create_fraud_alert(
    transaction_amount,
    merchant_category,
    payment_method,
    device_type,
    location,
    is_international,
    suspicious_keyword,
    fraud_probability,
    decision_threshold,
    prediction
):
    conn = sqlite3.connect(DB_PATH)

    query = """
    INSERT INTO fraud_alerts (
        created_at,
        transaction_amount,
        merchant_category,
        payment_method,
        device_type,
        location,
        is_international,
        suspicious_keyword,
        fraud_probability,
        decision_threshold,
        prediction,
        status
    )
    VALUES (
        datetime('now'),
        ?, ?, ?, ?, ?, ?, ?, ?, ?, ?,
        'Open'
    )
    """

    conn.execute(
        query,
        (
            transaction_amount,
            merchant_category,
            payment_method,
            device_type,
            location,
            is_international,
            suspicious_keyword,
            fraud_probability,
            decision_threshold,
            prediction
        )
    )

    conn.commit()
    conn.close()

    return "Fraud alert created successfully."

11 — Test Fraud Alert

In [17]:
result = create_fraud_alert(
    transaction_amount=500.00,
    merchant_category="Electronics",
    payment_method="UPI",
    device_type="Mobile",
    location="Delhi",
    is_international=1,
    suspicious_keyword="Yes",
    fraud_probability=0.9852,
    decision_threshold=0.70,
    prediction=1
)

print(result)

Fraud alert created successfully.


12 — Database mein verify karo

In [18]:
alerts = pd.read_sql_query(
    """
    SELECT *
    FROM fraud_alerts
    ORDER BY alert_id DESC
    """,
    sqlite3.connect(DB_PATH)
)

alerts

,alert_id,created_at,transaction_amount,merchant_category,payment_method,device_type,location,is_international,suspicious_keyword,fraud_probability,decision_threshold,prediction,status
0,1,2026-10-01 08:10:10,500.0,Electronics,UPI,Mobile,Delhi,1,Yes,0.9852,0.7,1,Open


13 — Count verify karo

In [19]:
alert_count = pd.read_sql_query(
    """
    SELECT COUNT(*) AS total_alerts
    FROM fraud_alerts
    """,
    sqlite3.connect(DB_PATH)
)

alert_count

,total_alerts
0,1


Cell 14 — Automatic Prediction + Alert

In [20]:
def predict_and_create_alert(
    transaction_amount,
    merchant_category,
    payment_method,
    device_type,
    location,
    is_international,
    suspicious_keyword,
    fraud_probability,
    decision_threshold,
    prediction
):
    """
    Creates a fraud alert only when the model prediction is fraud.
    """

    if prediction == 1 and fraud_probability >= decision_threshold:

        result = create_fraud_alert(
            transaction_amount=transaction_amount,
            merchant_category=merchant_category,
            payment_method=payment_method,
            device_type=device_type,
            location=location,
            is_international=is_international,
            suspicious_keyword=suspicious_keyword,
            fraud_probability=fraud_probability,
            decision_threshold=decision_threshold,
            prediction=prediction
        )

        return {
            "prediction": "Fraud",
            "fraud_probability": fraud_probability,
            "alert_created": True,
            "message": result
        }

    return {
        "prediction": "Normal",
        "fraud_probability": fraud_probability,
        "alert_created": False,
        "message": "No fraud alert created."
    }

STEP 15.4 — Test Automatic Fraud Alert

In [21]:
fraud_test = predict_and_create_alert(
    transaction_amount=750.00,
    merchant_category="Electronics",
    payment_method="UPI",
    device_type="Mobile",
    location="Delhi",
    is_international=1,
    suspicious_keyword="Yes",
    fraud_probability=0.95,
    decision_threshold=0.70,
    prediction=1
)

fraud_test

{'prediction': 'Fraud',
 'fraud_probability': 0.95,
 'alert_created': True,
 'message': 'Fraud alert created successfully.'}

STEP 15.5 — Normal Transaction Test

In [22]:
normal_test = predict_and_create_alert(
    transaction_amount=50.00,
    merchant_category="Grocery",
    payment_method="UPI",
    device_type="Mobile",
    location="Delhi",
    is_international=0,
    suspicious_keyword="No",
    fraud_probability=0.25,
    decision_threshold=0.70,
    prediction=0
)

normal_test

{'prediction': 'Normal',
 'fraud_probability': 0.25,
 'alert_created': False,
 'message': 'No fraud alert created.'}

STEP 15.6 — Database Verify

In [23]:
alerts = pd.read_sql_query(
    """
    SELECT
        alert_id,
        created_at,
        transaction_amount,
        merchant_category,
        payment_method,
        device_type,
        location,
        fraud_probability,
        decision_threshold,
        prediction,
        status
    FROM fraud_alerts
    ORDER BY alert_id DESC
    """,
    sqlite3.connect(DB_PATH)
)

alerts

,alert_id,created_at,transaction_amount,merchant_category,payment_method,device_type,location,fraud_probability,decision_threshold,prediction,status
0,2,2026-10-01 08:16:53,750.0,Electronics,UPI,Mobile,Delhi,0.9500,0.7,1,Open
1,1,2026-10-01 08:10:10,500.0,Electronics,UPI,Mobile,Delhi,0.9852,0.7,1,Open


STEP 15.7 — Alert Statistics

In [24]:
alert_summary = pd.read_sql_query(
    """
    SELECT
        COUNT(*) AS total_alerts,
        SUM(CASE WHEN status = 'Open' THEN 1 ELSE 0 END) AS open_alerts,
        SUM(CASE WHEN prediction = 1 THEN 1 ELSE 0 END) AS fraud_alerts,
        ROUND(AVG(fraud_probability) * 100, 2) AS avg_fraud_probability
    FROM fraud_alerts
    """,
    sqlite3.connect(DB_PATH)
)

alert_summary

,total_alerts,open_alerts,fraud_alerts,avg_fraud_probability
0,2,2,2,96.76
